In [1]:
import sys
import os
sys.path.append(os.path.dirname(os.getcwd()))
from shared.collect_data import DataCollector
from shared.clean_data import DataCleaning
import ta
import matplotlib.pyplot as plt
from pandas.plotting import scatter_matrix
import seaborn as sns
import pandas as pd
import numpy as np

In [8]:
data_collector = DataCollector(
    symbol = "BTC/USDT", 
    timeframe = "15m", 
    days = 3,
    limit = 1000
)
df = data_collector.collect_data()

In [9]:
df.tail()

,timestamp,open,high,low,close,volume
283,2025-10-10 02:15:00,121272.50,121401.21,121242.01,121349.64,86.81548
284,2025-10-10 02:30:00,121349.65,121367.04,121066.55,121220.95,204.31662
285,2025-10-10 02:45:00,121220.96,121277.20,120900.00,121059.73,281.94034
286,2025-10-10 03:00:00,121059.73,121108.22,120909.15,121092.03,133.96251
287,2025-10-10 03:15:00,121092.03,121165.75,121092.02,121150.01,40.61421


In [10]:
df = df.iloc[:-1]

In [11]:
df.tail()

,timestamp,open,high,low,close,volume
282,2025-10-10 02:00:00,121623.96,121651.33,121227.23,121272.50,239.17874
283,2025-10-10 02:15:00,121272.50,121401.21,121242.01,121349.64,86.81548
284,2025-10-10 02:30:00,121349.65,121367.04,121066.55,121220.95,204.31662
285,2025-10-10 02:45:00,121220.96,121277.20,120900.00,121059.73,281.94034
286,2025-10-10 03:00:00,121059.73,121108.22,120909.15,121092.03,133.96251


In [12]:
##Awesome Oscillator (AO)
df["AO"] = ta.momentum.AwesomeOscillatorIndicator(
    high=df["high"],
    low=df["low"],
    window1=5,
    window2=34
).awesome_oscillator()

##KAMA
df["KAMA"] = ta.momentum.KAMAIndicator(close=df['close']).kama()

## PPO
ppo = ta.momentum.PercentagePriceOscillator(
    close=df['close'],
    window_slow=26,
    window_fast=12,
    window_sign=9
)

df["PPO"] = ppo.ppo()

df["PPO_HIST"] = ppo.ppo_hist()

df["PPO_SIGNAL"] = ppo.ppo_signal()

## PVO
pvo = ta.momentum.PercentageVolumeOscillator(
    volume =df['volume'],
    window_slow=26,
    window_fast=12,
    window_sign=9
)

df["PVO"] = pvo.pvo()

df["PVO_HIST"] = pvo.pvo_hist()

df["PVO_SIGNAL"] = pvo.pvo_signal()

## ROC
df["ROC"] = ta.momentum.ROCIndicator(
    close=df['close'],
    window=12
).roc()

## RSI
df["RSI"] = ta.momentum.RSIIndicator(close=df['close'], window=14).rsi()

## STOCHASTIC RSI
df["STOCH_RSI"] = ta.momentum.StochRSIIndicator(
    close=df['close'],
    window=14,
    smooth1=3,
    smooth2=3
).stochrsi()

## STOCHASTIC OSCILLATOR
stoch = ta.momentum.StochasticOscillator(
    high=df['high'],
    low=df['low'],
    close=df['close'],
    window=14,
    smooth_window=3
)

df["STOCH_OSC"] = stoch.stoch()

df["STOCH_OSC_SIGNAL"] = stoch.stoch_signal()

## TSI
df["TSI"] = ta.momentum.TSIIndicator(
    close=df['close'],
    window_slow=25,
    window_fast=13
).tsi()

In [13]:
## ADI
adi = ta.volume.AccDistIndexIndicator(
    high=df['high'],
    low=df['low'],
    close=df['close'],
    volume=df['volume']
)

df["ADI"] = adi.acc_dist_index()

## CMF
cmf = ta.volume.ChaikinMoneyFlowIndicator(
    high=df['high'],
    low=df['low'],
    close=df['close'],
    volume=df['volume'],
    window=20
)

df["CMF"] = cmf.chaikin_money_flow()

##EoM EMV
emi = ta.volume.EaseOfMovementIndicator(
    high=df["high"], low=df["low"], volume=df["volume"], window=14
)

df["EoM"] = emi.ease_of_movement()
df["EMV"] = emi.sma_ease_of_movement()

##FI
fi = ta.volume.ForceIndexIndicator(
    close=df["close"], volume=df["volume"], window=13
)

df["FI"] = fi.force_index()

##MFI
mfi = ta.volume.MFIIndicator(
    high=df["high"], low=df["low"], close=df["close"], volume=df["volume"], window=14
)

df["MFI"] = mfi.money_flow_index()

##NVI
nvi = ta.volume.NegativeVolumeIndexIndicator(
    close=df["close"], volume=df["volume"]
)

df["NVI"] = nvi.negative_volume_index()

##OBV
obv = ta.volume.OnBalanceVolumeIndicator(
    close=df["close"], volume=df["volume"]
)

df["OBV"] = obv.on_balance_volume()

##VPT
vpt = ta.volume.VolumePriceTrendIndicator(
    close=df["close"], volume=df["volume"]
)

df["VPT"] = vpt.volume_price_trend()

## VWAP
vwap = ta.volume.VolumeWeightedAveragePrice(
    high=df["high"],
    low=df["low"],
    close=df["close"],
    volume=df["volume"],
    window=14
)

df["VWAP"] = vwap.volume_weighted_average_price()

In [14]:
## ATR
atr = ta.volatility.AverageTrueRange(
    high=df["high"],
    low=df["low"],
    close=df["close"],
    window=14,
)

df["ATR"] = atr.average_true_range()

## BBANDS
bbands = ta.volatility.BollingerBands(
    close=df["close"],
    window=20,
    window_dev=2
)

df["BBANDS_HIGH"] = bbands.bollinger_hband()
df["BBANDS_HIGH_INDI"] = bbands.bollinger_hband_indicator()
df["BBANDS_LOWER"] = bbands.bollinger_lband()
df["BBANDS_LOWER_INDI"] = bbands.bollinger_lband_indicator()
df["BBANDS_MIDDLE"] = bbands.bollinger_mavg()
df["BBANDS_CHANNEL"] = bbands.bollinger_pband()
df["BBANDS_WIDTH"] = bbands.bollinger_wband()

## DONCHIAN CHANNEL
donchian = ta.volatility.DonchianChannel(
    high=df["high"],
    low=df["low"],
    close=df["close"],
    window=20
)

df["DONCHIAN_HIGH"] = donchian.donchian_channel_hband()
df["DONCHIAN_LOW"] = donchian.donchian_channel_lband()
df["DONCHIAN_MIDDLE"] = donchian.donchian_channel_mband()
df["DONCHIAN_PERCENTAGE"] = donchian.donchian_channel_pband()
df["DONCHIAN_WIDTH"] = donchian.donchian_channel_wband()

## KELTNER CHANNEL
keltner = ta.volatility.KeltnerChannel(
    high=df["high"],
    low=df["low"],
    close=df["close"],
    window=20,
    window_atr=10,
)

df["KELTNER_HIGH"] = keltner.keltner_channel_hband()
df["KELTNER_HIGH_INDI"] = keltner.keltner_channel_hband_indicator()
df["KELTNER_LOW"] = keltner.keltner_channel_lband()
df["KELTNER_LOW_INDI"] = keltner.keltner_channel_lband_indicator()
df["KELTNER_MIDDLE"] = keltner.keltner_channel_mband()
df["KELTNER_PERCENTAGE"] = keltner.keltner_channel_pband()
df["KELTNER_WIDTH"] = keltner.keltner_channel_wband()

## Ulcer Index
ui = ta.volatility.UlcerIndex(
    close=df["close"],
    window=14
)

df["ULCER_INDEX"] = ui.ulcer_index()

In [15]:
## ADX
adx = ta.trend.ADXIndicator(
    high=df["high"],
    low=df["low"],
    close=df["close"],
    window=14,
)

df["ADX"] = adx.adx()
df["ADX_POS"] = adx.adx_pos()
df["ADX_NEG"] = adx.adx_neg()

## AROON
aroon = ta.trend.AroonIndicator(
    high=df["high"],
    low=df["low"],
    window=25
)

df["AROON_UP"] = aroon.aroon_up()
df["AROON_DOWN"] = aroon.aroon_down()
df["AROON_INDI"] = aroon.aroon_indicator()

## CCI
cci = ta.trend.CCIIndicator(
    high=df["high"],
    low=df["low"],
    close=df["close"],
    window=20
)

df["CCI"] = cci.cci()

## DPO
dpo = ta.trend.DPOIndicator(
    close=df["close"],
    window=20
)

df["DPO"] = dpo.dpo()

## EMA
ema = ta.trend.EMAIndicator(
    close=df["close"],
    window=14
)

df["EMA"] = ema.ema_indicator()

## KST
kst = ta.trend.KSTIndicator(
    close=df["close"],
    roc1=10,
    roc2=15,
    roc3=20,
    roc4=30,
    window1=10,
    window2=10,
    window3=10,
    window4=15,
    nsig =9
)

df["KST"] = kst.kst()
df["KST_SIGNAL"] = kst.kst_sig()
df["KST_DIFF"] = kst.kst_diff()

## MACD
macd = ta.trend.MACD(
    close=df["close"],
    window_slow=26,
    window_fast=12,
    window_sign=9
)

df["MACD"] = macd.macd()
df["MACD_SIGNAL"] = macd.macd_signal()
df["MACD_DIFF"] = macd.macd_diff()

## MI
mi = ta.trend.MassIndex(
    high=df["high"],
    low=df["low"],
    window_slow=25,
    window_fast=9
)

df["MASS_INDEX"] = mi.mass_index()

## SMA
sma = ta.trend.SMAIndicator(
    close=df["close"],
    window=14
)

df["SMA"] = sma.sma_indicator()

## VI
vi = ta.trend.VortexIndicator(
    high=df["high"],
    low=df["low"],
    close=df["close"],
    window=14
)

df["VI_POS"] = vi.vortex_indicator_pos()
df["VI_NEG"] = vi.vortex_indicator_neg()
df["VI_DIFF"] = vi.vortex_indicator_diff()

## WMA
wma = ta.trend.WMAIndicator(
    close=df["close"],
    window=14
)

df["WMA"] = wma.wma()

In [16]:
df.head()

,timestamp,open,high,low,close,volume,AO,KAMA,PPO,PPO_HIST,...,KST_DIFF,MACD,MACD_SIGNAL,MACD_DIFF,MASS_INDEX,SMA,VI_POS,VI_NEG,VI_DIFF,WMA
0,2025-10-07 03:30:00,124152.08,124294.35,124101.26,124268.55,99.93911,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-10-07 03:45:00,124268.55,124329.74,124246.23,124329.74,56.00028,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-10-07 04:00:00,124329.74,124329.74,124140.86,124175.31,140.54336,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-10-07 04:15:00,124175.31,124424.01,124109.10,124424.01,76.83962,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-10-07 04:30:00,124424.01,124654.47,124398.96,124591.02,129.24320,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
df.tail()

,timestamp,open,high,low,close,volume,AO,KAMA,PPO,PPO_HIST,...,KST_DIFF,MACD,MACD_SIGNAL,MACD_DIFF,MASS_INDEX,SMA,VI_POS,VI_NEG,VI_DIFF,WMA
282,2025-10-10 02:00:00,121623.96,121651.33,121227.23,121272.50,239.17874,351.086882,121582.563314,0.070083,-0.025933,...,0.458470,85.150244,116.637446,-31.487202,22.142145,121633.781429,0.916452,1.009539,-0.093086,121632.710190
283,2025-10-10 02:15:00,121272.50,121401.21,121242.01,121349.64,86.81548,237.521029,121571.616959,0.049511,-0.037204,...,-0.148495,60.150413,105.340040,-45.189627,22.413207,121609.742143,0.865168,1.120302,-0.255134,121594.824667
284,2025-10-10 02:30:00,121349.65,121367.04,121066.55,121220.95,204.31662,126.090324,121548.899034,0.024379,-0.049869,...,-0.771083,29.612307,90.194493,-60.582186,22.772014,121592.657857,0.880655,1.102959,-0.222304,121542.985714
285,2025-10-10 02:45:00,121220.96,121277.20,120900.00,121059.73,281.94034,3.396676,121454.842422,-0.006186,-0.064347,...,-1.421735,-7.511877,70.653219,-78.165096,23.231796,121560.717143,0.865565,1.078849,-0.213284,121471.928667
286,2025-10-10 03:00:00,121059.73,121108.22,120909.15,121092.03,133.96251,-133.662765,121402.513720,-0.027951,-0.068890,...,-1.781683,-33.935532,49.735469,-83.671001,23.544286,121529.745714,0.847134,1.123436,-0.276301,121409.437048


In [19]:
import json

all_features = {}
for hour in [1, 2, 3, 4]:
    # Load the JSON file
    hyperparameters = {}
    if hour == 1:
        with open('feature_selection/features_by_target.json', 'r') as f:
            hyperparameters = json.load(f)
    else:
        with open(f'feature_selection/features_by_target_{hour}.json', 'r') as f:
            hyperparameters = json.load(f)

    all_features.update(hyperparameters)

In [26]:
all_features

{'close_step_1': ['low',
  'DONCHIAN_LOW',
  'DONCHIAN_MIDDLE',
  'KAMA',
  'WMA',
  'BBANDS_MIDDLE',
  'close',
  'KELTNER_MIDDLE',
  'dayofyear',
  'EMA',
  'close_step_5'],
 'close_step_2': ['low',
  'DONCHIAN_LOW',
  'KAMA',
  'DONCHIAN_MIDDLE',
  'WMA',
  'KELTNER_MIDDLE',
  'BBANDS_MIDDLE',
  'close',
  'EMA',
  'dayofyear',
  'close_step_5',
  'KELTNER_LOW',
  'VWAP'],
 'close_step_3': ['low',
  'DONCHIAN_LOW',
  'KAMA',
  'BBANDS_MIDDLE',
  'close_step_5',
  'DONCHIAN_MIDDLE',
  'KELTNER_MIDDLE',
  'WMA',
  'EMA',
  'dayofyear',
  'KELTNER_LOW',
  'close',
  'close_step_6',
  'VWAP',
  'BBANDS_LOWER'],
 'close_step_4': ['DONCHIAN_LOW',
  'low',
  'close_step_6',
  'BBANDS_MIDDLE',
  'KELTNER_MIDDLE',
  'close_step_5',
  'KAMA',
  'DONCHIAN_MIDDLE',
  'WMA',
  'KELTNER_LOW',
  'EMA',
  'BBANDS_LOWER',
  'close'],
 'close_step_5': ['low',
  'DONCHIAN_LOW',
  'KELTNER_MIDDLE',
  'BBANDS_MIDDLE',
  'KAMA',
  'EMA',
  'WMA',
  'DONCHIAN_MIDDLE',
  'close_step_9',
  'close',
  'dayof

In [22]:
import joblib
import json

In [27]:
for hour in range(1, 17):
    data_frame = df.copy()
    data_frame[all_features[f'close_step_{hour}']]
    print(data_frame)

KeyError: "['dayofyear', 'close_step_5'] not in index"

In [ ]:
model = joblib.load(f'hyper_parameter_tuning/xgb_model_close_step_{hour}.json')